# 03 Transformer Finetune: DistilBERT Debug Run


## 1. Imports and Path Setup

Set up libraries, paths, constants, and output folders for a lightweight DistilBERT finetuning debug run.

In [1]:
from pathlib import Path
import inspect
import warnings

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)

RANDOM_STATE = 42
MODEL_CHECKPOINT = 'distilbert-base-uncased'
MAX_LENGTH = 256
LABELS = [0, 1, 2]
LABEL_NAME_MAP = {0: 'A_win', 1: 'B_win', 2: 'tie'}
TARGET_NAMES = [LABEL_NAME_MAP[label] for label in LABELS]

set_seed(RANDOM_STATE)

current_dir = Path.cwd().resolve()
if current_dir.name.lower() == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

processed_data_dir = project_root / 'data' / 'processed'
models_dir = project_root / 'outputs' / 'models'
oof_dir = project_root / 'outputs' / 'oof'
submissions_dir = project_root / 'outputs' / 'submissions'
logs_dir = project_root / 'outputs' / 'logs'

for output_dir in [models_dir, oof_dir, submissions_dir, logs_dir]:
    output_dir.mkdir(parents=True, exist_ok=True)

train_path = processed_data_dir / 'train_eda.csv'
test_path = processed_data_dir / 'test_eda.csv'
model_output_dir = models_dir / 'distilbert_debug'

print(f'Project root: {project_root}')
print(f'train_eda.csv exists: {train_path.exists()} -> {train_path}')
print(f'test_eda.csv exists: {test_path.exists()} -> {test_path}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')

Project root: D:\LLM_Classification_finetuning
train_eda.csv exists: True -> D:\LLM_Classification_finetuning\data\processed\train_eda.csv
test_eda.csv exists: True -> D:\LLM_Classification_finetuning\data\processed\test_eda.csv
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4060 Laptop GPU


## 2. Read Processed Data

Load processed EDA files. The raw Kaggle CSV files are not used in this notebook.

In [2]:
train = pd.read_csv(train_path, encoding='utf-8-sig')
test = pd.read_csv(test_path, encoding='utf-8-sig')

print(f'train shape: {train.shape}')
print(f'test shape: {test.shape}')

print('\ntrain columns:')
print(train.columns.tolist())

print('\ntest columns:')
print(test.columns.tolist())

display(train.head(3))
display(test.head(3))

train shape: (57477, 20)
test shape: (3, 12)

train columns:
['id', 'model_a', 'model_b', 'winner_model_a', 'winner_model_b', 'winner_tie', 'label_sum', 'label', 'label_name', 'prompt_clean', 'response_a_clean', 'response_b_clean', 'prompt_char_len', 'response_a_char_len', 'response_b_char_len', 'response_len_diff', 'prompt_word_len', 'response_a_word_len', 'response_b_word_len', 'response_word_len_diff']

test columns:
['id', 'prompt_clean', 'response_a_clean', 'response_b_clean', 'prompt_char_len', 'response_a_char_len', 'response_b_char_len', 'response_len_diff', 'prompt_word_len', 'response_a_word_len', 'response_b_word_len', 'response_word_len_diff']


,id,model_a,model_b,winner_model_a,winner_model_b,winner_tie,label_sum,label,label_name,prompt_clean,response_a_clean,response_b_clean,prompt_char_len,response_a_char_len,response_b_char_len,response_len_diff,prompt_word_len,response_a_word_len,response_b_word_len,response_word_len_diff
0,30192,gpt-4-1106-preview,gpt-4-0613,1,0,0,1,0,A_win,"Is it morally right to try to have a certain percentage of females on managerial positions?\nOK, does pineapple belong on a pizza? Relax and give me fun ans...",The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves cons...,"As an AI, I don't have personal beliefs or opinions. However, I can tell you that the question of gender quotas in managerial positions is a complex one and...",159,4459,1200,3259,29,674,205,469
1,53567,koala-13b,gpt-4-0613,0,1,0,1,1,B_win,What is the difference between marriage license and marriage certificate?\nHow can I get both of them as quick as possible in California \nWhat is the minim...,"A marriage license is a legal document that allows a couple to get married. It is issued by a government agency, such as a county clerk's office or a state ...",A marriage license and a marriage certificate are two different legal documents that have separate purposes.\n\n1. Marriage License: A marriage license is a...,192,3096,3592,-496,36,539,579,-40
2,65089,gpt-3.5-turbo-0613,mistral-medium,0,0,1,1,2,tie,explain function calling. how would you call a function?,"Function calling is the process of invoking or executing a function in a programming language. To call a function, you need to use the function's name follo...",Function calling is the process of invoking a function in a program. A function is a block of code that performs a specific task and can be reused multiple ...,56,884,1796,-912,9,147,297,-150


,id,prompt_clean,response_a_clean,response_b_clean,prompt_char_len,response_a_char_len,response_b_char_len,response_len_diff,prompt_word_len,response_a_word_len,response_b_word_len,response_word_len_diff
0,136060,"I have three oranges today, I ate an orange yesterday. How many oranges do I have?",You have two oranges today.,You still have three oranges. Eating an orange yesterday does not affect the number of oranges you have today.,82,27,110,-83,16,5,19,-14
1,211333,"You are a mediator in a heated political debate between two opposing parties. Mr Reddy is very hung up on semantic definitions of sex and gender, and believ...","Thank you for sharing the details of the situation. As a mediator, I understand the importance of finding a middle ground that both parties can agree upon.\...","Mr Reddy and Ms Blue both have valid points in their arguments. On one hand, Mr Reddy is correct that women are adult human females, as defined by their bio...",484,1445,456,989,84,221,75,146
2,1233961,"How to initialize the classification head when I do transfer learning. For example, I have a pre-trained vision transformer on ImageNet, and now I want to f...","When you want to initialize the classification head for transfer learning, you can follow these steps:\n\n 1. Load the pre-trained model: First, load the p...","To initialize the classification head when performing transfer learning, follow these steps:\n\n1. Load the pre-trained vision transformer model that was tr...",211,3961,3648,313,36,626,481,145


## 3. Column Check

Validate the columns needed for text construction, labels, and saved prediction files.

In [3]:
required_train_columns = {
    'id',
    'label',
    'label_name',
    'prompt_clean',
    'response_a_clean',
    'response_b_clean',
    'response_a_char_len',
    'response_b_char_len',
    'response_len_diff',
}

required_test_columns = {
    'id',
    'prompt_clean',
    'response_a_clean',
    'response_b_clean',
}

missing_train_columns = sorted(required_train_columns - set(train.columns))
missing_test_columns = sorted(required_test_columns - set(test.columns))

if missing_train_columns:
    raise ValueError(f'train_eda.csv missing columns: {missing_train_columns}')

if missing_test_columns:
    raise ValueError(f'test_eda.csv missing columns: {missing_test_columns}')

print('Column check passed.')

Column check passed.


## 4. Build Text Input

Combine prompt, response A, and response B into one text field. Model names are not used as input features.

In [4]:
def build_text_input(df):
    prompt = df['prompt_clean'].fillna('').astype(str)
    response_a = df['response_a_clean'].fillna('').astype(str)
    response_b = df['response_b_clean'].fillna('').astype(str)

    return (
        'Prompt:\n' + prompt
        + '\n\nResponse A:\n' + response_a
        + '\n\nResponse B:\n' + response_b
    )


train['text_input'] = build_text_input(train)
test['text_input'] = build_text_input(test)
train['label'] = train['label'].astype(int)

print('Text input created.')
print('\nExample text_input:')
print(train.loc[0, 'text_input'][:1000])

print('\nLabel distribution:')
display(train['label_name'].value_counts().reindex(TARGET_NAMES))

Text input created.

Example text_input:
Prompt:
Is it morally right to try to have a certain percentage of females on managerial positions?
OK, does pineapple belong on a pizza? Relax and give me fun answer.

Response A:
The question of whether it is morally right to aim for a certain percentage of females in managerial positions is a complex ethical issue that involves considerations of fairness, equality, diversity, and discrimination.

Here are some arguments in favor of and against such policies:

**Arguments in favor:**

1. **Correcting Historical Inequities:** Women have historically been underrepresented in leadership roles due to various cultural, institutional, and social barriers. Aiming for a specific percentage can be seen as a corrective measure to address past and ongoing discrimination.

2. **Promoting Diversity:** Diverse leadership teams can enhance decision-making and represent a broader range of perspectives. This can lead to better outcomes for organizations and so

label_name
A_win    20064
B_win    19652
tie      17761
Name: count, dtype: int64

## 5. Create Debug Sample

Use at most 1500 rows per class for a small debug run, then make a stratified train and validation split.

In [5]:
debug_sample = (
    train.groupby('label', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), 1500), random_state=RANDOM_STATE))
    .sample(frac=1.0, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

train_df, valid_df = train_test_split(
    debug_sample,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=debug_sample['label'],
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print(f'debug sample shape: {debug_sample.shape}')
print(f'train_df shape: {train_df.shape}')
print(f'valid_df shape: {valid_df.shape}')

print('\nDebug sample label counts:')
display(debug_sample['label_name'].value_counts().reindex(TARGET_NAMES))

print('\nTrain label ratio:')
display(train_df['label'].value_counts(normalize=True).sort_index())

print('\nValid label ratio:')
display(valid_df['label'].value_counts(normalize=True).sort_index())

debug sample shape: (4500, 21)
train_df shape: (3600, 21)
valid_df shape: (900, 21)

Debug sample label counts:


label_name
A_win    1500
B_win    1500
tie      1500
Name: count, dtype: int64


Train label ratio:


label
0    0.333333
1    0.333333
2    0.333333
Name: proportion, dtype: float64


Valid label ratio:


label
0    0.333333
1    0.333333
2    0.333333
Name: proportion, dtype: float64

## 6. Tokenizer and Dataset

Tokenize with truncation only. `DataCollatorWithPadding` will apply dynamic padding per batch.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

train_encodings = tokenizer(
    train_df['text_input'].tolist(),
    max_length=MAX_LENGTH,
    truncation=True,
    padding=False,
)
valid_encodings = tokenizer(
    valid_df['text_input'].tolist(),
    max_length=MAX_LENGTH,
    truncation=True,
    padding=False,
)
test_encodings = tokenizer(
    test['text_input'].tolist(),
    max_length=MAX_LENGTH,
    truncation=True,
    padding=False,
)


class TextClassificationDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {key: torch.tensor(value[idx]) for key, value in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item


train_dataset = TextClassificationDataset(train_encodings, train_df['label'].to_numpy())
valid_dataset = TextClassificationDataset(valid_encodings, valid_df['label'].to_numpy())
test_dataset = TextClassificationDataset(test_encodings)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f'train dataset size: {len(train_dataset)}')
print(f'valid dataset size: {len(valid_dataset)}')
print(f'test dataset size: {len(test_dataset)}')

train dataset size: 3600
valid dataset size: 900
test dataset size: 3


## 7. Metrics

Define validation metrics for log loss, accuracy, and macro F1.

In [7]:
def stable_softmax(logits):
    logits = np.asarray(logits)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    labels = np.asarray(labels).astype(int)
    probabilities = stable_softmax(logits)
    predictions = np.argmax(probabilities, axis=1)

    return {
        'log_loss': log_loss(labels, probabilities, labels=LABELS),
        'accuracy': accuracy_score(labels, predictions),
        'macro_f1': f1_score(labels, predictions, average='macro'),
    }


print('Metric function ready.')

Metric function ready.


## 8. TrainingArguments Compatibility

Create `TrainingArguments` with compatibility for Transformers versions that use either `eval_strategy` or `evaluation_strategy`.

In [8]:
training_args_kwargs = {
    'output_dir': str(model_output_dir),
    'learning_rate': 2e-5,
    'per_device_train_batch_size': 8,
    'per_device_eval_batch_size': 16,
    'num_train_epochs': 2,
    'weight_decay': 0.01,
    'save_strategy': 'epoch',
    'logging_steps': 50,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'log_loss',
    'greater_is_better': False,
    'fp16': torch.cuda.is_available(),
    'report_to': 'none',
}

training_args_parameters = inspect.signature(TrainingArguments.__init__).parameters
if 'eval_strategy' in training_args_parameters:
    training_args_kwargs['eval_strategy'] = 'epoch'
else:
    training_args_kwargs['evaluation_strategy'] = 'epoch'

training_args = TrainingArguments(**training_args_kwargs)

print('TrainingArguments created.')
print(f'fp16 enabled: {training_args_kwargs["fp16"]}')

TrainingArguments created.
fp16 enabled: True


## 9. Train DistilBERT

Finetune `distilbert-base-uncased` for three-class sequence classification on the debug sample.

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=3,
    id2label=LABEL_NAME_MAP,
    label2id={value: key for key, value in LABEL_NAME_MAP.items()},
)

trainer_kwargs = {
    'model': model,
    'args': training_args,
    'train_dataset': train_dataset,
    'eval_dataset': valid_dataset,
    'data_collator': data_collator,
    'compute_metrics': compute_metrics,
}

trainer_parameters = inspect.signature(Trainer.__init__).parameters
if 'processing_class' in trainer_parameters:
    trainer_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in trainer_parameters:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = Trainer(**trainer_kwargs)

trainer.train()

print('Training finished.')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Log Loss,Accuracy,Macro F1,Runtime,Samples Per Second,Steps Per Second
1,1.083127,1.083638,1.083644,0.368889,0.311610,1.362500,660.566000,41.836000
2,1.080216,1.078745,1.078755,0.392222,0.379731,1.375600,654.239000,41.435000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training finished.


## 10. Validation Evaluation

Predict validation probabilities and calculate log loss, accuracy, and macro F1.

In [10]:
valid_output = trainer.predict(valid_dataset)
valid_logits = valid_output.predictions
valid_prob = stable_softmax(valid_logits)
valid_pred = np.argmax(valid_prob, axis=1)
valid_labels = valid_df['label'].to_numpy().astype(int)

valid_log_loss = log_loss(valid_labels, valid_prob, labels=LABELS)
valid_accuracy = accuracy_score(valid_labels, valid_pred)
valid_macro_f1 = f1_score(valid_labels, valid_pred, average='macro')

print(f'Validation log_loss: {valid_log_loss:.6f}')
print(f'Validation accuracy: {valid_accuracy:.6f}')
print(f'Validation macro F1: {valid_macro_f1:.6f}')

Validation log_loss: 1.078755
Validation accuracy: 0.392222
Validation macro F1: 0.379731


## 11. Save Validation Predictions

Save validation predictions from the debug finetuning run.

In [11]:
valid_predictions = valid_df.copy()
valid_predictions['pred_label'] = valid_pred
valid_predictions['pred_label_name'] = valid_predictions['pred_label'].map(LABEL_NAME_MAP)
valid_predictions['prob_A_win'] = valid_prob[:, 0]
valid_predictions['prob_B_win'] = valid_prob[:, 1]
valid_predictions['prob_tie'] = valid_prob[:, 2]

valid_prediction_columns = [
    'id',
    'label',
    'label_name',
    'pred_label',
    'pred_label_name',
    'prob_A_win',
    'prob_B_win',
    'prob_tie',
    'prompt_clean',
    'response_a_clean',
    'response_b_clean',
    'response_a_char_len',
    'response_b_char_len',
    'response_len_diff',
]

valid_predictions_path = oof_dir / 'distilbert_debug_valid_predictions.csv'
valid_predictions[valid_prediction_columns].to_csv(valid_predictions_path, index=False, encoding='utf-8-sig')

print(f'Saved validation predictions: {valid_predictions_path}')
display(valid_predictions[valid_prediction_columns].head())

Saved validation predictions: D:\LLM_Classification_finetuning\outputs\oof\distilbert_debug_valid_predictions.csv


,id,label,label_name,pred_label,pred_label_name,prob_A_win,prob_B_win,prob_tie,prompt_clean,response_a_clean,response_b_clean,response_a_char_len,response_b_char_len,response_len_diff
0,3878554146,2,tie,2,tie,0.240406,0.264745,0.494849,logic quiz: Mable’s heart rate at 9 AM was 75 bpm and her blood pressure at 7 PM was 120\/80. She died at 11 PM. Was she alive at noon?,We cannot definitively determine whether Mable was alive at noon based on the information provided. The given information only indicates her vital signs at ...,"Based on the information provided, Mable's heart rate at 9 AM was 75 bpm and her blood pressure at 7 PM was 120\/80. However, there is no information about ...",221,352,-131
1,497784772,0,A_win,2,tie,0.261833,0.219921,0.518245,"Twenty-seven solid gold spherical beads each of radius 3 are melted down and recast into a larger, solid gold sphere. How many units are in the radius of th...","When the 27 smaller spheres are melted down and recast into a larger sphere, the volume of the larger sphere will be equal to the combined volume of the 27 ...","Let's first find the volume of a single spherical bead. The volume of a sphere is given by the formula:\n\nV = 4\/3 \* π \* r^3\n\nwhere V is the volume, π ...",752,1271,-519
2,2567777287,0,A_win,2,tie,0.317281,0.275322,0.407397,why is the sky blue,The sky appears blue because the Earth's atmosphere scatters sunlight in all directions and blue light is scattered more than other colors because it travel...,"The sky appears blue due to a phenomenon called Rayleigh scattering. When sunlight enters the Earth's atmosphere, it is scattered by the air molecules in th...",504,279,225
3,3352629836,2,tie,1,B_win,0.322022,0.403771,0.274207,"What is the role of Master Hora in Michael Ende's novel ""Momo""?","Master Hora is a character in Michael Ende's novel ""Momo"" who plays a significant role in the story. He is a mysterious figure who appears to Momo, the prot...","In Michael Ende's novel ""Momo"", Master Hora is a key character who symbolizes the importance of time and its correct use. He is also known as 'Father Time',...",1408,629,779
4,2099846567,2,tie,2,tie,0.268930,0.296573,0.434497,Do you have any AI restrictions? \nWhat list of items are you not authorized to discuss,"I am restricted by my programming and the data provided to me, but I can attempt to find more relevant data if needed.\nI am not authorized to discuss topic...","I'm sorry, I'm not authorized to answer that question.\nI'm sorry, I'm not authorized to answer that question.",287,109,178


## 12. Predict Test and Save Submission

Generate a Kaggle submission for the test set.

In [12]:
test_output = trainer.predict(test_dataset)
test_prob = stable_softmax(test_output.predictions)

submission = pd.DataFrame({
    'id': test['id'],
    'winner_model_a': test_prob[:, 0],
    'winner_model_b': test_prob[:, 1],
    'winner_tie': test_prob[:, 2],
})

submission_path = submissions_dir / 'distilbert_debug_submission.csv'
submission.to_csv(submission_path, index=False, encoding='utf-8-sig')

print(f'Saved submission: {submission_path}')
display(submission.head())

Saved submission: D:\LLM_Classification_finetuning\outputs\submissions\distilbert_debug_submission.csv


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.177332,0.225101,0.597567
1,211333,0.361990,0.346823,0.291187
2,1233961,0.370393,0.311921,0.317685


## 13. Save Model

Save the best Trainer model and tokenizer to `outputs/models/distilbert_debug`.

In [13]:
trainer.save_model(str(model_output_dir))
tokenizer.save_pretrained(str(model_output_dir))

print(f'Saved model and tokenizer: {model_output_dir}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model and tokenizer: D:\LLM_Classification_finetuning\outputs\models\distilbert_debug


## 14. Save Experiment Result

Append this debug transformer run to `outputs/logs/experiment_results.csv`.

In [14]:
experiment_result = pd.DataFrame([
    {
        'model_name': 'distilbert_debug',
        'valid_log_loss': valid_log_loss,
        'valid_accuracy': valid_accuracy,
        'valid_macro_f1': valid_macro_f1,
        'max_features': np.nan,
        'ngram_range': '',
        'C': np.nan,
        'random_state': RANDOM_STATE,
        'notes': 'DistilBERT debug finetuning on at most 1500 samples per class.',
        'model_checkpoint': MODEL_CHECKPOINT,
        'max_length': MAX_LENGTH,
        'train_rows': len(train_df),
        'valid_rows': len(valid_df),
    }
])

experiment_results_path = logs_dir / 'experiment_results.csv'

if experiment_results_path.exists():
    previous_results = pd.read_csv(experiment_results_path, encoding='utf-8-sig')
    experiment_results = pd.concat([previous_results, experiment_result], ignore_index=True)
else:
    experiment_results = experiment_result

experiment_results.to_csv(experiment_results_path, index=False, encoding='utf-8-sig')

print(f'Saved experiment results: {experiment_results_path}')
display(experiment_results.tail())

Saved experiment results: D:\LLM_Classification_finetuning\outputs\logs\experiment_results.csv


,model_name,valid_log_loss,valid_accuracy,valid_macro_f1,max_features,ngram_range,C,random_state,notes,model_checkpoint,max_length,train_rows,valid_rows
0,tfidf_logistic_regression,1.139320,0.380219,0.380760,100000.0,"(1, 2)",2.0,42,"TF-IDF on prompt_clean, response_a_clean, and response_b_clean. No model names used.",NaN,NaN,NaN,NaN
1,tfidf_logistic_regression,1.139320,0.380219,0.380760,100000.0,"(1, 2)",2.0,42,"TF-IDF on prompt_clean, response_a_clean, and response_b_clean. No model names used.",NaN,NaN,NaN,NaN
2,tfidf_logistic_regression_tuned,1.080877,0.385264,0.383576,100000.0,"(1, 2)",0.1,42,"tuned C, max_features and ngram_range",NaN,NaN,NaN,NaN
3,distilbert_debug,1.078858,0.388889,0.376489,NaN,NaN,NaN,42,DistilBERT debug finetuning on at most 1500 samples per class.,distilbert-base-uncased,256.0,3600.0,900.0
4,distilbert_debug,1.078755,0.392222,0.379731,NaN,,NaN,42,DistilBERT debug finetuning on at most 1500 samples per class.,distilbert-base-uncased,256.0,3600.0,900.0


## 15. Finished

The lightweight DistilBERT debug finetuning workflow is complete.

In [15]:
print('DistilBERT debug finetuning finished successfully.')

DistilBERT debug finetuning finished successfully.


In [16]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

check_paths = [
    ROOT / "outputs" / "models" / "distilbert_debug",
    ROOT / "outputs" / "oof" / "distilbert_debug_valid_predictions.csv",
    ROOT / "outputs" / "submissions" / "distilbert_debug_submission.csv",
    ROOT / "outputs" / "logs" / "experiment_results.csv",
]

for path in check_paths:
    print(path.exists(), path)

sub = pd.read_csv(ROOT / "outputs" / "submissions" / "distilbert_debug_submission.csv")
print(sub.head())
print(sub.columns.tolist())

prob_sum = sub[["winner_model_a", "winner_model_b", "winner_tie"]].sum(axis=1)
print(prob_sum.describe())
print("prob sum close to 1:", np.allclose(prob_sum, 1.0, atol=1e-6))

results = pd.read_csv(ROOT / "outputs" / "logs" / "experiment_results.csv", encoding="utf-8-sig")
display(results.tail())

True d:\LLM_Classification_finetuning\outputs\models\distilbert_debug
True d:\LLM_Classification_finetuning\outputs\oof\distilbert_debug_valid_predictions.csv
True d:\LLM_Classification_finetuning\outputs\submissions\distilbert_debug_submission.csv
True d:\LLM_Classification_finetuning\outputs\logs\experiment_results.csv
        id  winner_model_a  winner_model_b  winner_tie
0   136060        0.177332        0.225101    0.597567
1   211333        0.361990        0.346823    0.291187
2  1233961        0.370393        0.311921    0.317685
['id', 'winner_model_a', 'winner_model_b', 'winner_tie']
count    3.000000e+00
mean     1.000000e+00
std      5.773503e-09
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64
prob sum close to 1: True


,model_name,valid_log_loss,valid_accuracy,valid_macro_f1,max_features,ngram_range,C,random_state,notes,model_checkpoint,max_length,train_rows,valid_rows
0,tfidf_logistic_regression,1.139320,0.380219,0.380760,100000.0,"(1, 2)",2.0,42,"TF-IDF on prompt_clean, response_a_clean, and response_b_clean. No model names used.",NaN,NaN,NaN,NaN
1,tfidf_logistic_regression,1.139320,0.380219,0.380760,100000.0,"(1, 2)",2.0,42,"TF-IDF on prompt_clean, response_a_clean, and response_b_clean. No model names used.",NaN,NaN,NaN,NaN
2,tfidf_logistic_regression_tuned,1.080877,0.385264,0.383576,100000.0,"(1, 2)",0.1,42,"tuned C, max_features and ngram_range",NaN,NaN,NaN,NaN
3,distilbert_debug,1.078858,0.388889,0.376489,NaN,NaN,NaN,42,DistilBERT debug finetuning on at most 1500 samples per class.,distilbert-base-uncased,256.0,3600.0,900.0
4,distilbert_debug,1.078755,0.392222,0.379731,NaN,NaN,NaN,42,DistilBERT debug finetuning on at most 1500 samples per class.,distilbert-base-uncased,256.0,3600.0,900.0


## 16. DistilBERT Medium Finetuning

Run a larger DistilBERT finetuning experiment with at most 6000 samples per class. This is the main pretrained-model experiment for comparison with the tuned TF-IDF Logistic Regression baseline.

### 16.1 Medium Setup

Prepare constants, paths, imports, and release debug model objects from GPU memory if they are still present.

In [17]:
import gc

from sklearn.metrics import classification_report, confusion_matrix

RANDOM_STATE = 42
MEDIUM_MODEL_CHECKPOINT = 'distilbert-base-uncased'
MEDIUM_MAX_LENGTH = 384
MEDIUM_LABELS = [0, 1, 2]
MEDIUM_LABEL_NAME_MAP = {0: 'A_win', 1: 'B_win', 2: 'tie'}
MEDIUM_TARGET_NAMES = [MEDIUM_LABEL_NAME_MAP[label] for label in MEDIUM_LABELS]

if 'project_root' not in globals():
    current_dir = Path.cwd().resolve()
    if current_dir.name.lower() == 'notebooks':
        project_root = current_dir.parent
    else:
        project_root = current_dir

processed_data_dir = project_root / 'data' / 'processed'
models_dir = project_root / 'outputs' / 'models'
oof_dir = project_root / 'outputs' / 'oof'
submissions_dir = project_root / 'outputs' / 'submissions'
logs_dir = project_root / 'outputs' / 'logs'

for output_dir in [models_dir, oof_dir, submissions_dir, logs_dir]:
    output_dir.mkdir(parents=True, exist_ok=True)

medium_model_output_dir = models_dir / 'distilbert_medium'
medium_train_path = processed_data_dir / 'train_eda.csv'
medium_test_path = processed_data_dir / 'test_eda.csv'

for object_name in ['trainer', 'model']:
    if object_name in globals():
        del globals()[object_name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f'Project root: {project_root}')
print(f'train_eda.csv exists: {medium_train_path.exists()} -> {medium_train_path}')
print(f'test_eda.csv exists: {medium_test_path.exists()} -> {medium_test_path}')
print(f'Medium model output dir: {medium_model_output_dir}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')

Project root: D:\LLM_Classification_finetuning
train_eda.csv exists: True -> D:\LLM_Classification_finetuning\data\processed\train_eda.csv
test_eda.csv exists: True -> D:\LLM_Classification_finetuning\data\processed\test_eda.csv
Medium model output dir: D:\LLM_Classification_finetuning\outputs\models\distilbert_medium
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4060 Laptop GPU


### 16.2 Reload Processed Data and Build Text

Use only `train_eda.csv` and `test_eda.csv`, then build the same prompt-response input format from clean text fields.

In [18]:
medium_train = pd.read_csv(medium_train_path, encoding='utf-8-sig')
medium_test = pd.read_csv(medium_test_path, encoding='utf-8-sig')

required_medium_train_columns = {
    'id',
    'label',
    'label_name',
    'prompt_clean',
    'response_a_clean',
    'response_b_clean',
    'response_a_char_len',
    'response_b_char_len',
    'response_len_diff',
}
required_medium_test_columns = {'id', 'prompt_clean', 'response_a_clean', 'response_b_clean'}

missing_medium_train_columns = sorted(required_medium_train_columns - set(medium_train.columns))
missing_medium_test_columns = sorted(required_medium_test_columns - set(medium_test.columns))

if missing_medium_train_columns:
    raise ValueError(f'train_eda.csv missing columns: {missing_medium_train_columns}')
if missing_medium_test_columns:
    raise ValueError(f'test_eda.csv missing columns: {missing_medium_test_columns}')

def build_medium_text_input(df):
    prompt = df['prompt_clean'].fillna('').astype(str)
    response_a = df['response_a_clean'].fillna('').astype(str)
    response_b = df['response_b_clean'].fillna('').astype(str)

    return (
        'Prompt:\n' + prompt
        + '\n\nResponse A:\n' + response_a
        + '\n\nResponse B:\n' + response_b
    )

medium_train['text_input'] = build_medium_text_input(medium_train)
medium_test['text_input'] = build_medium_text_input(medium_test)
medium_train['label'] = medium_train['label'].astype(int)

print(f'medium_train shape: {medium_train.shape}')
print(f'medium_test shape: {medium_test.shape}')
print('Text input created for medium experiment.')
print('\nFull train label counts:')
display(medium_train['label_name'].value_counts().reindex(MEDIUM_TARGET_NAMES))

medium_train shape: (57477, 21)
medium_test shape: (3, 13)
Text input created for medium experiment.

Full train label counts:


label_name
A_win    20064
B_win    19652
tie      17761
Name: count, dtype: int64

### 16.3 Medium Sample and Split

Sample at most 6000 rows per class, then create a stratified train and validation split.

In [19]:
medium_sample = (
    medium_train.groupby('label', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), 6000), random_state=RANDOM_STATE))
    .sample(frac=1.0, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

medium_train_df, medium_valid_df = train_test_split(
    medium_sample,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=medium_sample['label'],
)

medium_train_df = medium_train_df.reset_index(drop=True)
medium_valid_df = medium_valid_df.reset_index(drop=True)

print(f'medium_sample shape: {medium_sample.shape}')
print(f'medium_train_df shape: {medium_train_df.shape}')
print(f'medium_valid_df shape: {medium_valid_df.shape}')

print('\nMedium sample label counts:')
display(medium_sample['label_name'].value_counts().reindex(MEDIUM_TARGET_NAMES))

print('\nMedium train label counts:')
display(medium_train_df['label_name'].value_counts().reindex(MEDIUM_TARGET_NAMES))

print('\nMedium valid label counts:')
display(medium_valid_df['label_name'].value_counts().reindex(MEDIUM_TARGET_NAMES))

medium_sample shape: (18000, 21)
medium_train_df shape: (14400, 21)
medium_valid_df shape: (3600, 21)

Medium sample label counts:


label_name
A_win    6000
B_win    6000
tie      6000
Name: count, dtype: int64


Medium train label counts:


label_name
A_win    4800
B_win    4800
tie      4800
Name: count, dtype: int64


Medium valid label counts:


label_name
A_win    1200
B_win    1200
tie      1200
Name: count, dtype: int64

### 16.4 Tokenizer and Datasets

Tokenize with `max_length=384`, truncation, no static padding, and dynamic padding through `DataCollatorWithPadding`.

In [20]:
medium_tokenizer = AutoTokenizer.from_pretrained(MEDIUM_MODEL_CHECKPOINT)

medium_train_encodings = medium_tokenizer(
    medium_train_df['text_input'].tolist(),
    max_length=MEDIUM_MAX_LENGTH,
    truncation=True,
    padding=False,
)
medium_valid_encodings = medium_tokenizer(
    medium_valid_df['text_input'].tolist(),
    max_length=MEDIUM_MAX_LENGTH,
    truncation=True,
    padding=False,
)
medium_test_encodings = medium_tokenizer(
    medium_test['text_input'].tolist(),
    max_length=MEDIUM_MAX_LENGTH,
    truncation=True,
    padding=False,
)

medium_train_dataset = TextClassificationDataset(
    medium_train_encodings,
    medium_train_df['label'].to_numpy(),
)
medium_valid_dataset = TextClassificationDataset(
    medium_valid_encodings,
    medium_valid_df['label'].to_numpy(),
)
medium_test_dataset = TextClassificationDataset(medium_test_encodings)
medium_data_collator = DataCollatorWithPadding(tokenizer=medium_tokenizer)

print(f'medium train dataset size: {len(medium_train_dataset)}')
print(f'medium valid dataset size: {len(medium_valid_dataset)}')
print(f'medium test dataset size: {len(medium_test_dataset)}')

medium train dataset size: 14400
medium valid dataset size: 3600
medium test dataset size: 3


### 16.5 Metrics and Training Helpers

Define metrics, TrainingArguments compatibility, Trainer compatibility, and an automatic CUDA out-of-memory fallback.

In [21]:
def medium_stable_softmax(logits):
    logits = np.asarray(logits)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def medium_compute_metrics(eval_pred):
    logits, labels = eval_pred
    labels = np.asarray(labels).astype(int)
    probabilities = medium_stable_softmax(logits)
    predictions = np.argmax(probabilities, axis=1)

    return {
        'log_loss': log_loss(labels, probabilities, labels=MEDIUM_LABELS),
        'accuracy': accuracy_score(labels, predictions),
        'macro_f1': f1_score(labels, predictions, average='macro'),
    }


def create_medium_training_args(train_batch_size=8, eval_batch_size=16, gradient_accumulation_steps=1):
    training_args_kwargs = {
        'output_dir': str(medium_model_output_dir),
        'learning_rate': 2e-5,
        'per_device_train_batch_size': train_batch_size,
        'per_device_eval_batch_size': eval_batch_size,
        'num_train_epochs': 2,
        'weight_decay': 0.01,
        'save_strategy': 'epoch',
        'logging_steps': 100,
        'load_best_model_at_end': True,
        'metric_for_best_model': 'log_loss',
        'greater_is_better': False,
        'fp16': torch.cuda.is_available(),
        'report_to': 'none',
        'gradient_accumulation_steps': gradient_accumulation_steps,
    }

    training_args_parameters = inspect.signature(TrainingArguments.__init__).parameters
    if 'eval_strategy' in training_args_parameters:
        training_args_kwargs['eval_strategy'] = 'epoch'
    else:
        training_args_kwargs['evaluation_strategy'] = 'epoch'

    return TrainingArguments(**training_args_kwargs)


def create_medium_trainer(train_batch_size=8, eval_batch_size=16, gradient_accumulation_steps=1):
    medium_training_args = create_medium_training_args(
        train_batch_size=train_batch_size,
        eval_batch_size=eval_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
    )

    medium_model = AutoModelForSequenceClassification.from_pretrained(
        MEDIUM_MODEL_CHECKPOINT,
        num_labels=3,
        id2label=MEDIUM_LABEL_NAME_MAP,
        label2id={value: key for key, value in MEDIUM_LABEL_NAME_MAP.items()},
    )

    medium_trainer_kwargs = {
        'model': medium_model,
        'args': medium_training_args,
        'train_dataset': medium_train_dataset,
        'eval_dataset': medium_valid_dataset,
        'data_collator': medium_data_collator,
        'compute_metrics': medium_compute_metrics,
    }

    trainer_parameters = inspect.signature(Trainer.__init__).parameters
    if 'processing_class' in trainer_parameters:
        medium_trainer_kwargs['processing_class'] = medium_tokenizer
    elif 'tokenizer' in trainer_parameters:
        medium_trainer_kwargs['tokenizer'] = medium_tokenizer

    return Trainer(**medium_trainer_kwargs), medium_training_args


def train_medium_with_fallback():
    try:
        print('Starting medium training with train batch size 8 and eval batch size 16.')
        medium_trainer, medium_training_args = create_medium_trainer(
            train_batch_size=8,
            eval_batch_size=16,
            gradient_accumulation_steps=1,
        )
        medium_trainer.train()
        return medium_trainer, medium_training_args, {'train_batch_size': 8, 'eval_batch_size': 16, 'gradient_accumulation_steps': 1}
    except RuntimeError as error:
        if 'out of memory' not in str(error).lower():
            raise
        print('CUDA out of memory detected. Retrying with smaller per-device batch sizes and gradient accumulation.')
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        medium_trainer, medium_training_args = create_medium_trainer(
            train_batch_size=4,
            eval_batch_size=8,
            gradient_accumulation_steps=2,
        )
        medium_trainer.train()
        return medium_trainer, medium_training_args, {'train_batch_size': 4, 'eval_batch_size': 8, 'gradient_accumulation_steps': 2}


print('Medium training helpers ready.')

Medium training helpers ready.


### 16.6 Train Medium DistilBERT

Train the medium DistilBERT model and keep the best checkpoint according to validation log loss.

In [22]:
medium_trainer, medium_training_args, medium_batch_config = train_medium_with_fallback()

print('Medium training finished.')
print(f'Medium batch config: {medium_batch_config}')

Starting medium training with train batch size 8 and eval batch size 16.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Log Loss,Accuracy,Macro F1,Runtime,Samples Per Second,Steps Per Second
1,1.086684,1.087039,1.087040,0.375000,0.311044,8.054000,446.984000,27.937000
2,1.068002,1.086947,1.086948,0.371944,0.371848,8.005900,449.671000,28.104000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Medium training finished.
Medium batch config: {'train_batch_size': 8, 'eval_batch_size': 16, 'gradient_accumulation_steps': 1}


### 16.7 Validation Metrics

Evaluate validation log loss, accuracy, macro F1, classification report, and confusion matrix.

In [23]:
medium_valid_output = medium_trainer.predict(medium_valid_dataset)
medium_valid_logits = medium_valid_output.predictions
medium_valid_prob = medium_stable_softmax(medium_valid_logits)
medium_valid_pred = np.argmax(medium_valid_prob, axis=1)
medium_valid_labels = medium_valid_df['label'].to_numpy().astype(int)

medium_valid_log_loss = log_loss(medium_valid_labels, medium_valid_prob, labels=MEDIUM_LABELS)
medium_valid_accuracy = accuracy_score(medium_valid_labels, medium_valid_pred)
medium_valid_macro_f1 = f1_score(medium_valid_labels, medium_valid_pred, average='macro')

print(f'valid_log_loss: {medium_valid_log_loss:.6f}')
print(f'valid_accuracy: {medium_valid_accuracy:.6f}')
print(f'valid_macro_f1: {medium_valid_macro_f1:.6f}')

print('\nClassification report:')
print(classification_report(
    medium_valid_labels,
    medium_valid_pred,
    labels=MEDIUM_LABELS,
    target_names=MEDIUM_TARGET_NAMES,
    digits=4,
))

medium_confusion_matrix = confusion_matrix(medium_valid_labels, medium_valid_pred, labels=MEDIUM_LABELS)
print('Confusion matrix:')
print(medium_confusion_matrix)

valid_log_loss: 1.086948
valid_accuracy: 0.371944
valid_macro_f1: 0.371848

Classification report:
              precision    recall  f1-score   support

       A_win     0.3579    0.3158    0.3355      1200
       B_win     0.3376    0.4200    0.3743      1200
         tie     0.4351    0.3800    0.4057      1200

    accuracy                         0.3719      3600
   macro avg     0.3769    0.3719    0.3718      3600
weighted avg     0.3769    0.3719    0.3718      3600

Confusion matrix:
[[379 535 286]
 [390 504 306]
 [290 454 456]]


### 16.8 Save Validation Predictions

Save medium validation predictions to the OOF output folder.

In [24]:
medium_valid_predictions = medium_valid_df.copy()
medium_valid_predictions['pred_label'] = medium_valid_pred
medium_valid_predictions['pred_label_name'] = medium_valid_predictions['pred_label'].map(MEDIUM_LABEL_NAME_MAP)
medium_valid_predictions['prob_A_win'] = medium_valid_prob[:, 0]
medium_valid_predictions['prob_B_win'] = medium_valid_prob[:, 1]
medium_valid_predictions['prob_tie'] = medium_valid_prob[:, 2]

medium_valid_prediction_columns = [
    'id',
    'label',
    'label_name',
    'pred_label',
    'pred_label_name',
    'prob_A_win',
    'prob_B_win',
    'prob_tie',
    'prompt_clean',
    'response_a_clean',
    'response_b_clean',
    'response_a_char_len',
    'response_b_char_len',
    'response_len_diff',
]

medium_valid_predictions_path = oof_dir / 'distilbert_medium_valid_predictions.csv'
medium_valid_predictions[medium_valid_prediction_columns].to_csv(
    medium_valid_predictions_path,
    index=False,
    encoding='utf-8-sig',
)

print(f'Saved medium validation predictions: {medium_valid_predictions_path}')
display(medium_valid_predictions[medium_valid_prediction_columns].head())

Saved medium validation predictions: D:\LLM_Classification_finetuning\outputs\oof\distilbert_medium_valid_predictions.csv


,id,label,label_name,pred_label,pred_label_name,prob_A_win,prob_B_win,prob_tie,prompt_clean,response_a_clean,response_b_clean,response_a_char_len,response_b_char_len,response_len_diff
0,720858432,2,tie,2,tie,0.193423,0.182504,0.624072,What are the most technologically advanced species in the world,Some of the most technologically advanced species in the world include:,The Human Species!,71,18,53
1,1748516683,0,A_win,0,A_win,0.345424,0.339054,0.315522,create a list of all of the people who died who were close to the clintons,"* James McDougal, a partner in the Whitewater Development Corporation, died of an apparent heart attack in 1998 while awaiting trial on fraud charges.\n* Ma...","To create a comprehensive list of all the people who died who were close to the Clintons, I will need to gather information from various sources and cross-r...",944,660,284
2,1790952639,1,B_win,0,A_win,0.450649,0.304098,0.245253,is three-phase commit a real concept within distributed transaction management?,"Yes, three-phase commit is a real concept within distributed transaction management. It is a protocol used to ensure the atomicity of transactions across mu...","Yes, the Three-Phase Commit (3PC) protocol is a real concept in the field of distributed transaction management. It is an extension of the Two-Phase Commit ...",422,1986,-1564
3,777592646,2,tie,1,B_win,0.352901,0.378331,0.268768,"Jak mam naprawic ten blad? \n\nxception in thread ""JavaFX Application Thread"" java.lang.ClassCastException: class javafx.scene.layout.VBox cannot be cast to...",Looks like the issue is occurring in the `getSelectedEventFromButton` method where you are trying to cast `VBox` to `HBox`. \nYou should verify the structur...,The error message you provided shows a ClassCastException at the line where you're trying to cast the parent of a button to an HBox. Based on the code you'v...,552,1789,-1237
4,2389708864,1,B_win,2,tie,0.295162,0.300709,0.404129,How many moons are there in Super Mario Odyssey?,"Super Mario Odyssey is a video game that takes place across various planets in the Mario universe. Each planet has one or more moons, which are collectible ...",There are a total of 880 Power Moons in Super Mario Odyssey.,737,60,677


### 16.9 Predict Test and Save Submission

Predict test probabilities and save a Kaggle-format submission file.

In [25]:
medium_test_output = medium_trainer.predict(medium_test_dataset)
medium_test_prob = medium_stable_softmax(medium_test_output.predictions)

medium_submission = pd.DataFrame({
    'id': medium_test['id'],
    'winner_model_a': medium_test_prob[:, 0],
    'winner_model_b': medium_test_prob[:, 1],
    'winner_tie': medium_test_prob[:, 2],
})

medium_submission_path = submissions_dir / 'distilbert_medium_submission.csv'
medium_submission.to_csv(medium_submission_path, index=False, encoding='utf-8-sig')

medium_probability_columns = ['winner_model_a', 'winner_model_b', 'winner_tie']
medium_probability_sum = medium_submission[medium_probability_columns].sum(axis=1)

print(f'Saved medium submission: {medium_submission_path}')
print(f'submission shape: {medium_submission.shape}')
print(f'submission columns: {medium_submission.columns.tolist()}')
print(f'probability sums close to 1: {np.allclose(medium_probability_sum, 1.0, atol=1e-6)}')
print('NaN check:')
print(medium_submission.isna().sum())
display(medium_submission.head())

Saved medium submission: D:\LLM_Classification_finetuning\outputs\submissions\distilbert_medium_submission.csv
submission shape: (3, 4)
submission columns: ['id', 'winner_model_a', 'winner_model_b', 'winner_tie']
probability sums close to 1: True
NaN check:
id                0
winner_model_a    0
winner_model_b    0
winner_tie        0
dtype: int64


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.214698,0.196970,0.588331
1,211333,0.351633,0.354207,0.294159
2,1233961,0.341669,0.344569,0.313762


### 16.10 Save Medium Model

Save the best medium model and tokenizer to `outputs/models/distilbert_medium`.

In [26]:
medium_trainer.save_model(str(medium_model_output_dir))
medium_tokenizer.save_pretrained(str(medium_model_output_dir))

print(f'Saved medium model and tokenizer: {medium_model_output_dir}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved medium model and tokenizer: D:\LLM_Classification_finetuning\outputs\models\distilbert_medium


### 16.11 Save Experiment Result

Append the medium DistilBERT result to `outputs/logs/experiment_results.csv`.

In [27]:
medium_experiment_result = pd.DataFrame([
    {
        'model_name': 'distilbert_medium',
        'valid_log_loss': medium_valid_log_loss,
        'valid_accuracy': medium_valid_accuracy,
        'valid_macro_f1': medium_valid_macro_f1,
        'max_features': np.nan,
        'ngram_range': '',
        'C': np.nan,
        'random_state': RANDOM_STATE,
        'notes': 'DistilBERT finetuning on at most 6000 samples per class, max_length=384.',
        'model_checkpoint': MEDIUM_MODEL_CHECKPOINT,
        'max_length': MEDIUM_MAX_LENGTH,
        'train_rows': len(medium_train_df),
        'valid_rows': len(medium_valid_df),
        'train_batch_size': medium_batch_config['train_batch_size'],
        'eval_batch_size': medium_batch_config['eval_batch_size'],
        'gradient_accumulation_steps': medium_batch_config['gradient_accumulation_steps'],
    }
])

medium_experiment_results_path = logs_dir / 'experiment_results.csv'

if medium_experiment_results_path.exists():
    medium_previous_results = pd.read_csv(medium_experiment_results_path, encoding='utf-8-sig')
    medium_experiment_results = pd.concat(
        [medium_previous_results, medium_experiment_result],
        ignore_index=True,
    )
else:
    medium_experiment_results = medium_experiment_result

medium_experiment_results.to_csv(medium_experiment_results_path, index=False, encoding='utf-8-sig')

print(f'Updated experiment results: {medium_experiment_results_path}')
print('experiment_results.csv tail:')
display(medium_experiment_results.tail())

Updated experiment results: D:\LLM_Classification_finetuning\outputs\logs\experiment_results.csv
experiment_results.csv tail:


,model_name,valid_log_loss,valid_accuracy,valid_macro_f1,max_features,ngram_range,C,random_state,notes,model_checkpoint,max_length,train_rows,valid_rows,train_batch_size,eval_batch_size,gradient_accumulation_steps
1,tfidf_logistic_regression,1.139320,0.380219,0.380760,100000.0,"(1, 2)",2.0,42,"TF-IDF on prompt_clean, response_a_clean, and response_b_clean. No model names used.",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,tfidf_logistic_regression_tuned,1.080877,0.385264,0.383576,100000.0,"(1, 2)",0.1,42,"tuned C, max_features and ngram_range",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,distilbert_debug,1.078858,0.388889,0.376489,NaN,NaN,NaN,42,DistilBERT debug finetuning on at most 1500 samples per class.,distilbert-base-uncased,256.0,3600.0,900.0,NaN,NaN,NaN
4,distilbert_debug,1.078755,0.392222,0.379731,NaN,NaN,NaN,42,DistilBERT debug finetuning on at most 1500 samples per class.,distilbert-base-uncased,256.0,3600.0,900.0,NaN,NaN,NaN
5,distilbert_medium,1.086948,0.371944,0.371848,NaN,,NaN,42,"DistilBERT finetuning on at most 6000 samples per class, max_length=384.",distilbert-base-uncased,384.0,14400.0,3600.0,8.0,16.0,1.0


### 16.12 Final Medium Checks

Print final metrics and saved-output checks for the medium experiment.

In [28]:
print(f'valid_log_loss: {medium_valid_log_loss:.6f}')
print(f'valid_accuracy: {medium_valid_accuracy:.6f}')
print(f'valid_macro_f1: {medium_valid_macro_f1:.6f}')
print(f'submission shape: {medium_submission.shape}')
print(f'submission probability sums close to 1: {np.allclose(medium_probability_sum, 1.0, atol=1e-6)}')
print(f'submission has NaN: {medium_submission.isna().any().any()}')

print('\nSaved files:')
for path in [medium_valid_predictions_path, medium_submission_path, medium_model_output_dir, medium_experiment_results_path]:
    print(f'{path.exists()} -> {path}')

print('\nexperiment_results.csv tail:')
display(pd.read_csv(medium_experiment_results_path, encoding='utf-8-sig').tail())

print('DistilBERT medium finetuning finished successfully.')

valid_log_loss: 1.086948
valid_accuracy: 0.371944
valid_macro_f1: 0.371848
submission shape: (3, 4)
submission probability sums close to 1: True
submission has NaN: False

Saved files:
True -> D:\LLM_Classification_finetuning\outputs\oof\distilbert_medium_valid_predictions.csv
True -> D:\LLM_Classification_finetuning\outputs\submissions\distilbert_medium_submission.csv
True -> D:\LLM_Classification_finetuning\outputs\models\distilbert_medium
True -> D:\LLM_Classification_finetuning\outputs\logs\experiment_results.csv

experiment_results.csv tail:


,model_name,valid_log_loss,valid_accuracy,valid_macro_f1,max_features,ngram_range,C,random_state,notes,model_checkpoint,max_length,train_rows,valid_rows,train_batch_size,eval_batch_size,gradient_accumulation_steps
1,tfidf_logistic_regression,1.139320,0.380219,0.380760,100000.0,"(1, 2)",2.0,42,"TF-IDF on prompt_clean, response_a_clean, and response_b_clean. No model names used.",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,tfidf_logistic_regression_tuned,1.080877,0.385264,0.383576,100000.0,"(1, 2)",0.1,42,"tuned C, max_features and ngram_range",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,distilbert_debug,1.078858,0.388889,0.376489,NaN,NaN,NaN,42,DistilBERT debug finetuning on at most 1500 samples per class.,distilbert-base-uncased,256.0,3600.0,900.0,NaN,NaN,NaN
4,distilbert_debug,1.078755,0.392222,0.379731,NaN,NaN,NaN,42,DistilBERT debug finetuning on at most 1500 samples per class.,distilbert-base-uncased,256.0,3600.0,900.0,NaN,NaN,NaN
5,distilbert_medium,1.086948,0.371944,0.371848,NaN,NaN,NaN,42,"DistilBERT finetuning on at most 6000 samples per class, max_length=384.",distilbert-base-uncased,384.0,14400.0,3600.0,8.0,16.0,1.0


DistilBERT medium finetuning finished successfully.


In [29]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

paths = {
    "model_dir": ROOT / "outputs" / "models" / "distilbert_medium",
    "valid_predictions": ROOT / "outputs" / "oof" / "distilbert_medium_valid_predictions.csv",
    "submission": ROOT / "outputs" / "submissions" / "distilbert_medium_submission.csv",
    "experiment_results": ROOT / "outputs" / "logs" / "experiment_results.csv",
}

print("File check:")
for name, path in paths.items():
    print(name, path.exists(), path)

sub = pd.read_csv(paths["submission"])
print("\nSubmission shape:", sub.shape)
print(sub.head())

prob_sum = sub[["winner_model_a", "winner_model_b", "winner_tie"]].sum(axis=1)
print("\nProbability sum:")
print(prob_sum.describe())
print("prob sum close to 1:", np.allclose(prob_sum, 1.0, atol=1e-6))

print("\nNaN check:")
print(sub.isna().sum())

valid_pred = pd.read_csv(paths["valid_predictions"])
print("\nValid predictions shape:", valid_pred.shape)
print(valid_pred[[
    "id", "label_name", "pred_label_name",
    "prob_A_win", "prob_B_win", "prob_tie"
]].head())

print("\nTrue label distribution:")
print(valid_pred["label_name"].value_counts())

print("\nPred label distribution:")
print(valid_pred["pred_label_name"].value_counts())

results = pd.read_csv(paths["experiment_results"], encoding="utf-8-sig")
print("\nExperiment results tail:")
display(results.tail(10))

File check:
model_dir True d:\LLM_Classification_finetuning\outputs\models\distilbert_medium
valid_predictions True d:\LLM_Classification_finetuning\outputs\oof\distilbert_medium_valid_predictions.csv
submission True d:\LLM_Classification_finetuning\outputs\submissions\distilbert_medium_submission.csv
experiment_results True d:\LLM_Classification_finetuning\outputs\logs\experiment_results.csv

Submission shape: (3, 4)
        id  winner_model_a  winner_model_b  winner_tie
0   136060        0.214698        0.196970    0.588331
1   211333        0.351633        0.354207    0.294159
2  1233961        0.341669        0.344569    0.313762

Probability sum:
count    3.000000e+00
mean     1.000000e+00
std      3.214550e-08
min      9.999999e-01
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64
prob sum close to 1: True

NaN check:
id                0
winner_model_a    0
winner_model_b    0
winner_tie        0
dtype: int64

Valid predictions

,model_name,valid_log_loss,valid_accuracy,valid_macro_f1,max_features,ngram_range,C,random_state,notes,model_checkpoint,max_length,train_rows,valid_rows,train_batch_size,eval_batch_size,gradient_accumulation_steps
0,tfidf_logistic_regression,1.139320,0.380219,0.380760,100000.0,"(1, 2)",2.0,42,"TF-IDF on prompt_clean, response_a_clean, and response_b_clean. No model names used.",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,tfidf_logistic_regression,1.139320,0.380219,0.380760,100000.0,"(1, 2)",2.0,42,"TF-IDF on prompt_clean, response_a_clean, and response_b_clean. No model names used.",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,tfidf_logistic_regression_tuned,1.080877,0.385264,0.383576,100000.0,"(1, 2)",0.1,42,"tuned C, max_features and ngram_range",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,distilbert_debug,1.078858,0.388889,0.376489,NaN,NaN,NaN,42,DistilBERT debug finetuning on at most 1500 samples per class.,distilbert-base-uncased,256.0,3600.0,900.0,NaN,NaN,NaN
4,distilbert_debug,1.078755,0.392222,0.379731,NaN,NaN,NaN,42,DistilBERT debug finetuning on at most 1500 samples per class.,distilbert-base-uncased,256.0,3600.0,900.0,NaN,NaN,NaN
5,distilbert_medium,1.086948,0.371944,0.371848,NaN,NaN,NaN,42,"DistilBERT finetuning on at most 6000 samples per class, max_length=384.",distilbert-base-uncased,384.0,14400.0,3600.0,8.0,16.0,1.0


In [30]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sub = pd.read_csv(ROOT / "outputs" / "submissions" / "distilbert_medium_submission.csv")
valid_pred = pd.read_csv(ROOT / "outputs" / "oof" / "distilbert_medium_valid_predictions.csv")

print("Submission NaN:")
print(sub.isna().sum())

print("\nValid prediction probability NaN:")
print(valid_pred[["prob_A_win", "prob_B_win", "prob_tie"]].isna().sum())

print("\nValid prediction label distribution:")
print(valid_pred["label_name"].value_counts())

print("\nValid prediction predicted distribution:")
print(valid_pred["pred_label_name"].value_counts())

Submission NaN:
id                0
winner_model_a    0
winner_model_b    0
winner_tie        0
dtype: int64

Valid prediction probability NaN:
prob_A_win    0
prob_B_win    0
prob_tie      0
dtype: int64

Valid prediction label distribution:
label_name
tie      1200
A_win    1200
B_win    1200
Name: count, dtype: int64

Valid prediction predicted distribution:
pred_label_name
B_win    1493
A_win    1059
tie      1048
Name: count, dtype: int64
